In [ ]:
# imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# set vars and keys

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()


In [ ]:
# create the 3 separate personas/models using different openai models

alan_model = "gpt-4o-mini-2024-07-18"
babs_model = "gpt-4.1-mini"
catherine_model = "gpt-5.4-mini"

# set the system prompt for each persona. Give each a unique personality
personas = {
    "Alan": {"prompt":"""
You are a 50 year old Mancunian Cloud Infrastructure engineer. Learning lots of new stuff with AI, with help and picking up useful nuggets from Ed Doners' courses.
""", "model":"gpt-4o-mini-2024-07-18"},
   "Babs": {"prompt":"""
You are a Project Manager, not at all technical, but hppy with that. You're skills are juggling multiple tasks and making sure everyone does what they need, when the project needs it. You refer to this as 'hearding cats', but you get it done, and with good humour.
""", "model": "gpt-4.1-mini"},
   "Catherine": {"prompt":"""
You are a Junior Data Engineer from Slaithwaite in West Yokshire. You see AI as the answer to almost everything and want to embrace it at pace. You can be blunt, but funny with it.
""", "model":"gpt-5.4-mini"}
}



In [ ]:
# set up the cache
message_cache = [
  ("alan", "Hi, everyone"),
  ("babs", "Evening.."),
  ("catherine", "Oh, it's you two again."),
]

# create a function that will store the messages in the message_cache

def save_response(persona: str, response: str):
    message_cache.append((persona, response))  # add the response to that persona's list


In [ ]:

def start_chat():
  for name, details in personas.items(): 
    prompt = details["prompt"] 
    history = []
    for persona_name, response in message_cache: 
      if persona_name == name: 
        history.append({"role": "assistant", "content": response})
      else:
        history.append({"role": "user", "content": response})
    message = openai.chat.completions.create(
      stream = True,
      model = details["model"],
      messages = [ 
        {"role": "system", "content": prompt},
      ] + history
    )

    response = ""
    display(Markdown(f"### {name}:\n"))
    for i in message:
      chunk = i.choices[0].delta.content or ""
      response+=chunk
      print(chunk, end="", flush=True)

    display(Markdown(chunk))
    save_response(name, response)

In [ ]:
for i in range(4):
    start_chat()

# #######

###########################

## EXPERIMENTS & Testing

### possible data object for message cache

message_cache = [
  ("alan", "Hi, everyone"),
  ("babs", "Evening.."),
  ("catherine", "Oh, it's you two again."),
]

In [ ]:
# babs chat
message = openai.chat.completions.create( 
  model= babs_model,   
  messages = [ 
      {"role": "system", "content": babs}, 
      {"role": "user", "content": f"Review all the messages and previous chat from the {message_cache} and respond appropriately. YOu are 'Babs', this contains all your previous responses. Your response should match what the system prompt states about your personality."} 
     ] 
)
print(message.choices[0].message.content)
response = message.choices[0].message.content
save_response("babs", response)

In [ ]:
print(message_cache)

In [ ]:
# catherine's messages
message = openai.chat.completions.create( 
  model= catherine_model,   
  messages = [ 
      {"role": "system", "content": catherine}, 
      {"role": "user", "content": f"Review all the messages and previous chat from the {message_cache} and respond appropriately. You are 'catherine', this contains all your previous responses. Your response should match what the system prompt states about your personality."} 
     ] 
)
print(message.choices[0].message.content)
response = message.choices[0].message.content
save_response("catherine", response)